# Proposed thesis model: joint DiT classifier with semantic consolidation
**14 October 2026.** Author-selected computational design supported by exploratory diagnostics; not a completed five-task solution. The historical frozen notebooks 00?14 remain preserved.

Run All on a supplied remote container displays authenticated retained evidence without Internet access, TensorFlow import, model construction or training. The adjacent assets are required. Saved outputs can be viewed anywhere; under project policy all notebook computation remains remote.

In [1]:
from pathlib import Path
import json
import sys


ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "files/notebooks/thesis/candidate_20261014.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from files.notebooks.thesis.candidate_20261014 import read_asset, remote_only


remote_only()
architecture = read_asset("architecture.json")
evidence = read_asset("evidence.json")
screen = read_asset("screen_result.json")


## Computational architecture
Input is CIFAR-10 RGB, 32 x 32. The wrapper owns standardization: raw real and generated images enter its preprocessing contract once. The CNN patchifier uses patch size 2, producing a 16 x 16 token grid at width 128 with additive fixed 2D sine/cosine positions. Time and class embeddings condition the implemented adaptive-normalization blocks. Diffusion horizon: 1,000.

The denoising backbone has **8 blocks, 4 attention heads, FFN expansion 4**. The classifier branch has **4 blocks, 4 heads, FFN expansion 4**. Feature aggregation is {"1": [-1]}; classifier connections are {"-1": [-1]}. These are native layer-ID maps, not additional external networks. Forced global average pooling overrides token-only readout flags. The classifier hidden MLP has ratio 1, **linear activation and dropout 0.5**. Its probabilities remain softmax outputs: linear changes the hidden activation, not normalization. Drop-path is zero; the refinement CNN is disabled.

The continual wrapper maintains seen-class mapping and expands outputs natively. The same persistent **raw joint DiTClassifier** performs classification and epsilon prediction for replay. EMA is disabled. A separate U-Net is an auxiliary noise teacher only; it does not generate replay instead of the joint network. Full generic options, source-level constructor defaults and implementation digests are stored in architecture.json.

## Training and inference
Fixed five-task schedule: [7,3] -> [2,8] -> [5,6] -> [9,4] -> [0,1], data seed 42. Screening seed is 101; three model seeds are reserved for finalists. A stratified 10% TRAIN holdout supplies DEV. Classification uses all seen classes without task ID. The reported diagnostic evaluates 2,000 DEV examples (500/class), not official test.

Joint training: Adam, constant LR **0.0009430051881577431**, batch **128**, clipnorm **5**. Coefficients: primary CE **0.00215**, epsilon MSE **1**, previous-classifier soft KD **0.0043** (temperature 2, old-class scope), noise KD **0.5**. Previous/current noise-teacher weights are 1/1; classifier-teacher weights are 1/0. Student CE and classifier KD use noisy input capped at timestep 64, null-only class conditioning and all rows. Null/timestep masking is off; denoising uses the full 1,000-step range. Task one has no previous-class teacher.

The best diagnostic additionally weights old rows **5 times inside primary training CE**, with per-batch normalization by the sum of weights. KD, denoising and inference are unchanged. This is a qualified diagnostic adapter, not a full-history YAML option; it balances aggregate CE mass for the 500-old/2,500-current pool.

Native prospective budgets are 100 epochs/task. Each task first trains a fresh current-class U-Net noise specialist on 9,000 permitted TRAIN rows, then the joint student on 2,500 current rows plus 500 generated old rows after task one. U-Net widths: 64/128/256; block depth 3; bottleneck width 384 / depth 2; GELU, batch normalization, skip connections, average-pool downsampling and transposed-convolution upsampling. Teacher AdamW LR **0.0009232173954634668**, weight decay **0.000010569549424860102**, batch 128. These native budgets differ from the best diagnostic, which restores task one and applies **240 additional joint updates** before semantic phases.

Replay uses the raw joint network: **50 DDIM steps, eta 0, CFG 2.929069189794846**, uniform selection, fixed total 500 old images. Inference uses clean timestep 0, primary-head weight 1, distillation-head weight 0 and no timestep ensemble. Sampling/postprocessing and later classifier preprocessing follow the wrapper contract once.

## Semantic procedure
This uses existing **RouteController/RoutePhase**, a TMCL-inspired adaptation, not the separate native TMCLTrainer VI/MI protocol. Acquisition freezes the main network while learning task/class modulators with contrastive and orthogonality objectives. Consolidation freezes the acquired target and modulators, updating permitted semantic weights through representation alignment.

The candidate uses **100 acquisition + 100 consolidation updates**, separate Adam LR **0.001**, batch **32**, **4 TMCL-style augmentation views**, temperature **0.1**, alignment **1**, orthogonality **1**, and consolidation CE **0**. Phase noise levels are zero. Modulators are retained during training; ordinary inference is unmodulated. The reported late fork builds a fresh four-class bank; it does not reconstruct two-task gate history.

Only extra consolidation CE is removed. Ordinary joint classifier CE remains active. This computational design is not evidence of a biological mechanism or an exact JDCL/TMCL reproduction.

In [2]:
rows = [("Before semantic phases", screen["before"]), ("CE + alignment, matched checkpoint", screen["matched_combined"]), ("Alignment only, selected", screen["after"])]
print("Single-seed exploratory DEV; all four seen classes; no task ID")
print(f"{'Condition':38s} {'Old %':>8} {'New %':>8} {'Mean %':>8} {'Forget pp':>10}")
for name, values in rows:
    print(f"{name:38s} {100 * values['old_accuracy']:8.1f} {100 * values['new_accuracy']:8.1f} {100 * values['four_class_accuracy']:8.1f} {100 * values['forgetting']:10.1f}")
print("Original result SHA256:", evidence["best_result_sha256"])


Single-seed exploratory DEV; all four seen classes; no task ID
Condition                                 Old %    New %   Mean %  Forget pp
Before semantic phases                     60.6     61.2     60.9       22.9
CE + alignment, matched checkpoint         16.7     88.3     52.5       66.8
Alignment only, selected                   60.6     66.0     63.3       22.9
Original result SHA256: afd70722ae59373bcec1d977bcce8e372e4943f1b7a152e89a128302632a1a2f


## Evidence and remaining work
Selected late-fork result: **60.6% old  / 66.0% new  / 63.3% mean**. Task-one accuracy was 83.5%, so forgetting is 22.9 percentage points. The matched CE+alignment fork gives 16.7% old  / 88.3% new  / 52.5% mean. This supports removing extra CE in this generated-replay setting; it does not isolate each ingredient of the overall recipe.

Real-TRAIN replay-oracle diagnostics rescue retention but mix realism, diversity and unseen-TRAIN coverage; they are not a deployable generator fix. Generated classifier self-agreement is not independent semantic quality. Freezing final-output updates did not rescue forgetting, and simple augmentation helped only partly.

No five-task superiority, paper-level equivalence, final test accuracy, confidence interval or three-seed result is inferred. Notebook 16 reproduces the three late phase forks. Notebook 17 prepares a fresh native screen with **old CE 1**, explicitly different from the old CE 5 diagnostic. Final five-task confirmation remains required.

In [3]:
print("Active constructor and wrapper configuration:")
print(json.dumps(architecture["architecture"], indent=2))
print("Semantic configuration:")
print(json.dumps(architecture["semantic"], indent=2))
print("Implementation source files:", len(architecture["source_sha256"]))


Active constructor and wrapper configuration:
{
  "name": "dit_classifier",
  "wrapper_name": "diffusion_classifier",
  "kwargs": {
    "channels": 3,
    "classifier_dropout_rate": 0.5,
    "classifier_mlp_activation_func": "linear",
    "classifier_mlp_ratio": 1,
    "classifier_only_cls_token": true,
    "classifier_only_distil_token": true,
    "clf_cls_token_type": "new_weight",
    "clf_connection_ids_dict": {
      "-1": [
        -1
      ]
    },
    "clf_depth": 4,
    "clf_droppath_rate": 0.0,
    "clf_mha_num_heads": 4,
    "clf_vit_block_mlp_ratio": 4.0,
    "conds_merger_type": "add",
    "depth": 8,
    "dim": 128,
    "dim_forced": true,
    "droppath_rate": 0.0,
    "feature_aggregation_ids_dict": {
      "1": [
        -1
      ]
    },
    "force_global_avg_pooling": true,
    "image_size": 32,
    "mha_num_heads": 4,
    "num_classes": null,
    "patch_size": 2,
    "patches_pos_embed_type": "2d_sincos",
    "patches_pos_merger_type": "add",
    "patchify_with_cnn":